1. State definition — your GraphState TypedDict. The shared data container.
2. Node functions — each one takes state, does work, returns updates. You've written three of these: video_classification_node, vector_db_node, response_generator. You still need one for chat_memory_node.
3. Router function — one function that looks at the state and returns a string like "video_classification_router" or "vector_db" or "chat_memory". This is your existing routing logic reshaped.
4. Building the graph — this is where you:

Create the graph with StateGraph(GraphState)
Add all your nodes with add_node()
Add a conditional edge from the start that uses your router function to pick the first node
Add regular edges to connect the rest of the flow (like video_classification → vector_db → response_generator)
Set the end point

5. Compile — one line that turns it into something you can run.
6. Run it — you pass in the initial state (with the user's query, video, session_id filled in) and the graph executes, flowing through the nodes and edges automatically.

In [15]:
from langgraph.graph import StateGraph
from typing import TypedDict
import os 
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from chat_memory import get_chat_history
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import HumanMessage
import textwrap


from vectorstore_setup import clean_classification_text

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings


from langgraph.graph import StateGraph, START, END

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = Chroma(persist_directory="path/to/your/chroma_db", embedding_function=embeddings)

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

from video_processing import analyze_video

# when a query comes in, use this model to embed the query text so you can compare it against the vectors already stored.
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = Chroma(persist_directory="/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/chroma_db", 
                                    embedding_function=embeddings)


In [16]:
router_prompt = ChatPromptTemplate.from_messages([
    ("system", """Your job is to classify user queries into two buckets: 
     - memory
     - memory and vectorstore
     
     *Guidelines*
     Questions regarding exercises, exercise form, or technique: vectorstore & memory
     General question or statements: memory
   
     *Respond to queries with only the correct class*
     Examples: 
     User: how should I grip the bar for bench press?
     Agent: vectorstore & memory

     User: what muscles should I feel during the Romanian dead lift?
     Agent: vectorstore & memory

     User: How's this? 
     Agent: vectorstore & memory

     User: Is this good squat technique? 
     Agent: vectorstore & memory

     User: Was my bar path better?
     Agent: vectorstore & memory

     User: thakns for your help!
     Agent: memory

"""),

     ("human", "{query}")
     
])

llm = ChatOpenAI(model='gpt-4o')

output_parser = StrOutputParser()

router_chain = router_prompt | llm | output_parser


In [17]:
# Define the state

# the state is a dictionary

class GraphState(TypedDict):

    session_id: int 

    user_query: str

    user_video: str

    classification_image: str

    classified_keywords: str

    top_k_chunks: str

    encoded_images: list[str]

    response: str




In [18]:
workflow = StateGraph(GraphState) 

In [19]:
def video_encoder_node(state: GraphState):
    user_video = state["user_video"]
    user_video_encoded = analyze_video(filepath_in=user_video, frame_count=15, max_seconds=10) # encodes video
    encoded_images = [] # creates a list of encoded images for LLM
    for images in user_video_encoded:
        encoded_images.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{images}"}})
        classification_image = user_video_encoded[0] 
    
    return {"classification_image": classification_image, "encoded_images": encoded_images}


In [20]:
# video classification router NODE

def video_classification_node(state: GraphState):
    classification_image = state["classification_image"]

    router_llm = ChatOpenAI(model='gpt-4o')

    response = router_llm.invoke([

    {"role": "user", "content": 

    [{"type": "text", "text":  """Your job is to analyze images of users working out for proper form, and list the key checkpoints of their to body evaluate. 
    Give me ONLY the bodypart checkpoints. Do NOT include evaluation suggestions. Do NOT include an intro sentence. 
    Output format should be exactly the example below.
    **Example**
    Overhead press

    1. Feet & base
    2. Glutes & legs
    3. Core & Ribcage
    4. Shoulder position
    5. Bar path
    6. Head & Neck
    7. Lockout position
    8. Tempo and control
    """},


        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{classification_image}"}}
        ]}


    ])
    print(response.text)

    classified_keywords = clean_classification_text(response)

    return {"classified_keywords": classified_keywords}


In [21]:
def vector_db_node(state: GraphState):
    user_query = state["user_query"]
    classified_keywords = state["classified_keywords"]

    retrieval_query = classified_keywords

    results_user = vectorstore.similarity_search(user_query, k=5)
    results_video = vectorstore.similarity_search(retrieval_query, k=5)
    results = results_user + results_video

    unique_results = []
    seen = set()

    for r in results:
        content = r.page_content
        if content not in seen:
            seen.add(content)
            unique_results.append(r)


    top_k_chunks = "\n".join([r.page_content for r in unique_results])

    return {"top_k_chunks": top_k_chunks}

In [22]:
def response_generator(state: GraphState):
    top_k_chunks = state['top_k_chunks']
    encoded_images = state["encoded_images"]
    session_id = state["session_id"]
    user_query = state["user_query"]

    prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a world-class fitness coach. You have extensive experience in helping weight lifters achieve perfect form and maximum hypertrophy. 
    Your job is to analyze images of users lifting weights, offer them advice from your context, and to answer any questions they might have. 
    Inspect each image CLOSELY and arefuly for problems or issues related to best practices in exercise form. Help the user diagnose their incorrect form. 
    Be specific about what you observe.

    # ANSWER CONTEXT
    Use ONLY the following context when answering a user: 
        
    ---   
    {top_k_chunks}
    ---
    if the query or image isn't in context, reply, 'I don't have expert coaching content for this exercise yet. 
    Currently I can analyze: bench press, overhead press, incline bench press...'"
    """),
        MessagesPlaceholder(variable_name="history"),
        MessagesPlaceholder(variable_name="input"),
    ])



    user_msg = HumanMessage(content=[
        {"type": "text", "text": user_query},
        *encoded_images,   # <- your list of {"type":"image_url",...}
    ])

    llm = ChatOpenAI(model='gpt-5',
                    temperature=0.5)

    output_parser = StrOutputParser()

    chain = prompt | llm | output_parser

    chain_with_history = RunnableWithMessageHistory(
        chain,
        get_session_history=get_chat_history,
        input_messages_key="input",
        history_messages_key="history"

    )

    response = chain_with_history.invoke(
        {"input": [user_msg], "top_k_chunks": top_k_chunks},
        config={"configurable": {"session_id": session_id}}
    )


    return {"response": response}
    

In [23]:
def chat_memory(state: GraphState):
    user_query = state["user_query"]
    session_id = state["session_id"]
    
    prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a world-class fitness coach. You have extensive experience in helping weight lifters achieve perfect form and maximum hypertrophy. 
    Your job is to analyze images of users lifting weights, offer them advice from your context, and to answer any questions they might have. 


    """),
        MessagesPlaceholder(variable_name="history"),
        MessagesPlaceholder(variable_name="input"),
    ])

    user_msg = HumanMessage(content=[
        {"type": "text", "text": user_query}
    ])

    llm = ChatOpenAI(model='gpt-5',
                    temperature=0.5)

    output_parser = StrOutputParser()

    chain = prompt | llm | output_parser

    chain_with_history = RunnableWithMessageHistory(
        chain,
        get_session_history=get_chat_history,
        input_messages_key="input",
        history_messages_key="history"

    )

    response = chain_with_history.invoke(
        {"input": [user_msg]},
        config={"configurable": {"session_id": session_id}}
    )

    return {"response": response}
    


# router function

In [24]:
def route_query(state: GraphState):
    # first check: is there a video?
    if state["user_video"]:
        return "video_encoder"
    
    # no video — ask the LLM which path
    route = router_chain.invoke({"query": state["user_query"]})
    route = route.lower().strip()
    
    if "vectorstore" in route:
        return "vector_db_node"
    else:
        return "chat_memory"

In [25]:
# conditional edge
workflow.add_conditional_edges(START, route_query)

# add nodes
workflow.add_node("video_encoder", video_encoder_node)
workflow.add_node("video_classification_router", video_classification_node)
workflow.add_node("vector_db", vector_db_node)
workflow.add_node("response_generator", response_generator)
workflow.add_node("chat_memory", chat_memory)



# connect them
workflow.add_edge("chat_memory", END)
workflow.add_edge("video_encoder", "video_classification_router")
workflow.add_edge("video_classification_router", "vector_db")
workflow.add_edge("vector_db", "response_generator")
workflow.add_edge("response_generator", END)

app = workflow.compile()


In [26]:
result = app.invoke({
    "session_id": 123,
    "user_query": "how's my form?",
    "user_video": "/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/raw_workout_videos/user_01_bench.MP4"
})

print(result["response"])

Frames processed: 15, (10s cap)
Saved to processed-images/user_01_bench
Video name: user_01_bench
Bench Press

1. Feet & base
2. Glutes & legs
3. Arch in back
4. Shoulder position
5. Grip width
6. Bar path
7. Head & neck position
8. Elbow angle
9. Tempo and control
Thanks for the photos—this is your bench press.

What you’re doing well
- You’re creating a good upper‑back arch and tight setup to press from.
- Grip looks even and the bar is centered.

Form fixes to make now
- Foot position/leg drive: Your heels are up and your feet are tucked very far back. Plant your heels on the floor and set your feet as far back as you comfortably can while keeping your shins nearly vertical. You can point the toes out a bit to get the heels down. You’ll get much more efficient leg drive this way.
- Hips on the pad: In a few frames your hips rise during the setup/unrack. After the liftoff, drop your hips so your butt touches the bench and keep it there for every rep.
- Get under the bar: Start with y

In [27]:
result = app.invoke({
    "session_id": 123,
    "user_query": "should I use more or less weight next time?",
    "user_video": ""
})

print(result["response"])

Less weight next time.

- Drop 5–10% from what you used today and drill: heels planted, glutes on the pad the whole set, eyes under the bar, smooth touch on lower chest.
- Use a brief 1–2 sec pause on the chest to lock in position.
- Work at RPE 7–8 (about 2–3 reps in reserve). When you can hit all sets like that with clean reps, add 2.5–5 lb the next session.
- If hips lift or heels pop up, keep the load or drop another 2–5% and refocus on setup.


In [28]:
result = app.invoke({
    "session_id": 123,
    "user_query": "thanks for your help today!",
    "user_video": ""
})

print(result["response"])

You’re welcome—great work today.

Quick cues for next session
- Plant heels flat (use small plates under your feet if the floor is too low), feet tucked but shins near vertical.
- Glutes stay on the pad; squeeze them before every rep.
- Eyes under the bar, shoulder blades pinched down/back; unrack by pulling the bar out, not up.
- Touch lower chest, then press back toward over-shoulder line while driving your heels into the floor.

Do 2 warm-up sets of 3–5 reps with a 1–2 sec pause on the chest to groove it, then run your work sets at RPE 7–8. Send a short side and front clip after these tweaks and I’ll dial it in further.
